# GNN + DistilBERT Music Genre Prediction

This notebook presents the final reproducible workflow for multimodal music genre classification using audio graphs and DistilBERT-based title representations.

## 1. Project Setup

In [1]:
# Project setup

import os
import random
import numpy as np
import pandas as pd
import torch

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Define project paths
project_root = "/content/drive/MyDrive/GNN_BERT_Music"

data_dir = os.path.join(project_root, "data")
raw_data_dir = os.path.join(data_dir, "raw")
processed_data_dir = os.path.join(data_dir, "processed")
examples_dir = os.path.join(data_dir, "examples")

models_dir = os.path.join(project_root, "models")
results_dir = os.path.join(project_root, "results")
src_dir = os.path.join(project_root, "src")
report_dir = os.path.join(project_root, "report")

# Set random seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create required directories if they do not exist
for directory in [
    data_dir,
    raw_data_dir,
    processed_data_dir,
    examples_dir,
    models_dir,
    results_dir,
    src_dir,
    report_dir,
]:
    os.makedirs(directory, exist_ok=True)

print("Project root:", project_root)
print("Device:", device)
print("Random seed:", SEED)
print("Project setup completed successfully.")

Mounted at /content/drive
Project root: /content/drive/MyDrive/GNN_BERT_Music
Device: cpu
Random seed: 42
Project setup completed successfully.


## 2. FMA-small Metadata and Official Split

In [3]:
# Locate the FMA metadata file in the current runtime and Google Drive

import os

search_roots = [
    "/content",
    "/content/drive/MyDrive/GNN_BERT_Music"
]

metadata_candidates = []

for search_root in search_roots:
    if not os.path.exists(search_root):
        continue

    for root, _, files in os.walk(search_root):
        for filename in files:
            if filename == "tracks.csv":
                metadata_candidates.append(
                    os.path.join(root, filename)
                )

print("Found tracks.csv files:")

if metadata_candidates:
    for path in metadata_candidates:
        print(path)
else:
    print("No tracks.csv found.")

Found tracks.csv files:
No tracks.csv found.


In [4]:
# Check whether the FMA-small audio dataset is available

audio_root = "/content/fma_small"

if os.path.exists(audio_root):
    audio_files = []

    for root, _, files in os.walk(audio_root):
        for filename in files:
            if filename.lower().endswith(".mp3"):
                audio_files.append(
                    os.path.join(root, filename)
                )

    print("FMA-small directory found:", True)
    print("Number of MP3 files:", len(audio_files))
else:
    print("FMA-small directory found:", False)
    print("Path:", audio_root)

FMA-small directory found: False
Path: /content/fma_small


In [5]:
# Download and extract the FMA metadata

import os
import urllib.request
import zipfile

metadata_dir = "/content/fma_metadata"
metadata_zip = "/content/fma_metadata.zip"
metadata_url = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"

os.makedirs(metadata_dir, exist_ok=True)

if not os.path.exists(os.path.join(metadata_dir, "tracks.csv")):
    print("Downloading FMA metadata...")

    urllib.request.urlretrieve(
        metadata_url,
        metadata_zip
    )

    print("Download completed.")

    print("Extracting metadata...")

    with zipfile.ZipFile(metadata_zip, "r") as zip_ref:
        zip_ref.extractall("/content")

    print("Metadata extraction completed.")
else:
    print("FMA metadata already exists.")

print()
print("tracks.csv exists:",
      os.path.exists(os.path.join(metadata_dir, "tracks.csv")))

Download completed.
Extracting metadata...
Metadata extraction completed.

tracks.csv exists: True
